In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import gc
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import igraph as ig
import leidenalg as la
from dotenv import load_dotenv
import psycopg2
from sqlalchemy import create_engine, text
import time


env_file = Path("assemble-graph-database/.env")
if env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded environment variables from {env_file}")
else:
    print(f"⚠️  .env file not found at {env_file}")
    print("💡 Using default values")

# Configure display settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Libraries imported successfully")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Numpy version: {np.__version__}")

output_dir = Path("analysis_plots")
output_dir.mkdir(exist_ok=True)

✓ Loaded environment variables from assemble-graph-database/.env
✓ Libraries imported successfully
✓ Pandas version: 2.3.2
✓ Numpy version: 2.3.2


## Database Connection Setup

Connect to PostgreSQL database to load the graph data.

In [2]:
# PostgreSQL Database connection configuration

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'postgres')
DB_NAME = os.getenv('DB_NAME', 'steam_reviews')
DB_SSLMODE = os.getenv('DB_SSLMODE', 'disable')

# PostgreSQL connection string
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode={DB_SSLMODE}"

print(f"🐘 Connecting to PostgreSQL database:")
print(f"   Host: {DB_HOST}:{DB_PORT}")
print(f"   Database: {DB_NAME}")
print(f"   User: {DB_USER}")

# Connect to database
try:
    # Create SQLAlchemy engine for pandas integration
    engine = create_engine(connection_string)

    # Test connection with direct psycopg2
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME
    )

    print(f"✓ Connected to PostgreSQL database: {DB_HOST}:{DB_PORT}/{DB_NAME}")

    # Get database size information
    cursor = conn.cursor()
    cursor.execute("""
        SELECT pg_size_pretty(pg_database_size(%s)) as db_size;
    """, (DB_NAME,))
    db_size = cursor.fetchone()[0]
    print(f"✓ Database size: {db_size}")
    cursor.close()

    # For pandas compatibility, we'll use the engine for pd.read_sql_query
    # Note: conn is psycopg2 connection, engine is SQLAlchemy engine

except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print(f"💡 Make sure PostgreSQL is running and the database exists.")
    print(f"💡 Run: cd graph-analyser/assemble-graph-database && docker compose --env-file assemble-graph-database/.env up -d")
    raise

🐘 Connecting to PostgreSQL database:
   Host: 127.0.0.1:25432
   Database: steam_reviews
   User: postgres
✓ Connected to PostgreSQL database: 127.0.0.1:25432/steam_reviews
✓ Database size: 27 GB


## Load igraph into memory

Load nodes (games) and edges (game_projection_edges) from the PostgreSQL database.

In [12]:
print("Loading games (nodes) from database...")

# Load games table as nodes
games_query = """
SELECT 
    game_id,
    name,
    steam_appid,
    price_currency,
    price_final,
    price_initial,
    categories_ids,
    genre_ids,
    release_date,
    rating_dejus_rating,
    rating_dejus_required_age,
    degree,
    component_id,
    projection_degree,
    projection_weighted_degree,
    projection_harmonic_centrality,
    projection_betweenness_centrality,
    leiden_1_5_res_community,
    leiden_0_5_res_community
FROM games
ORDER BY games.projection_weighted_degree DESC
LIMIT 10000
"""

games_df = pd.read_sql(games_query, engine)

print(f"✓ Loaded {len(games_df):,} games (nodes)")
print(f"  Memory usage: {games_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nSample of games data:")
print(games_df.head())
print("\nGames data info:")
print(games_df.info())

Loading games (nodes) from database...
✓ Loaded 10,000 games (nodes)
  Memory usage: 6.36 MB

Sample of games data:
   game_id                    name  steam_appid price_currency  price_final  \
0   292030  Ведьмак 3: Дикая Охота       292030            KZT    1199900.0   
1  1086940         Baldur's Gate 3      1086940            PHP     259900.0   
2   377160               Fallout 4       377160            BRL       5999.0   
3  1091500          Cyberpunk 2077      1091500            BRL      19990.0   
4   242760              The Forest       242760            BRL       3799.0   

   price_initial                                     categories_ids  \
0      1199900.0  [2, 22, 29, 30, 64, 67, 66, 68, 78, 74, 79, 69...   
1       259900.0  [2, 1, 9, 38, 48, 27, 22, 28, 29, 64, 67, 66, ...   
2         5999.0    [2, 22, 28, 74, 79, 69, 70, 23, 41, 42, 43, 62]   
3        19990.0  [2, 22, 28, 29, 64, 67, 66, 68, 78, 74, 79, 69...   
4         3799.0              [2, 1, 9, 38, 48, 52, 53

In [15]:
print("Loading game projection edges from database...")

# Load game_projection_edges table
edges_query = """
SELECT 
    game_id_1,
    game_id_2,
    weight
FROM game_projection_edges
INNER JOIN games AS g1 ON game_projection_edges.game_id_1 = g1.game_id
INNER JOIN games AS g2 ON game_projection_edges.game_id_2 = g2.game_id
ORDER BY game_projection_edges.weight DESC
LIMIT 100000
"""

edges_df = pd.read_sql(edges_query, engine)

print(f"✓ Loaded {len(edges_df):,} edges")
print(f"  Memory usage: {edges_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nSample of edges data:")
print(edges_df.head())
print("\nEdge weight statistics:")
print(edges_df['weight'].describe())

Loading game projection edges from database...
✓ Loaded 100,000 edges
  Memory usage: 11.23 MB

Sample of edges data:
  game_id_1 game_id_2  weight
0   1245620    374320   71839
1   1091500    292030   64816
2   1086940   1245620   64010
3   1086940    292030   60402
4   1086940   1091500   57664

Edge weight statistics:
count    100000.000000
mean        944.743950
std        1459.651444
min         318.000000
25%         403.000000
50%         557.000000
75%         937.000000
max       71839.000000
Name: weight, dtype: float64


### Build igraph Graph

Construct the igraph object from the loaded nodes and edges.

In [16]:
print("Building igraph from database data...")

# Create a mapping from game_id to vertex index
game_id_to_idx = {game_id: idx for idx, game_id in enumerate(games_df['game_id'])}

# Create list of edges using vertex indices
edge_list = []
edge_weights = []

for _, row in edges_df.iterrows():
    game1_idx = game_id_to_idx.get(row['game_id_1'])
    game2_idx = game_id_to_idx.get(row['game_id_2'])
    
    if game1_idx is not None and game2_idx is not None:
        edge_list.append((game1_idx, game2_idx))
        edge_weights.append(row['weight'])

# Create the igraph Graph
g = ig.Graph(n=len(games_df), edges=edge_list, directed=False)

# Add edge weights
g.es['weight'] = edge_weights

# Add vertex attributes from games dataframe
for col in games_df.columns:
    g.vs[col] = games_df[col].tolist()

print(f"\n✓ Graph constructed successfully!")
print(f"  Vertices: {g.vcount():,}")
print(f"  Edges: {g.ecount():,}")
print(f"  Directed: {g.is_directed()}")
print(f"  Weighted: {'weight' in g.es.attributes()}")
print(f"\nVertex attributes: {g.vs.attributes()}")
print(f"Edge attributes: {g.es.attributes()}")

Building igraph from database data...

✓ Graph constructed successfully!
  Vertices: 10,000
  Edges: 99,999
  Directed: False
  Weighted: True

Vertex attributes: ['game_id', 'name', 'steam_appid', 'price_currency', 'price_final', 'price_initial', 'categories_ids', 'genre_ids', 'release_date', 'rating_dejus_rating', 'rating_dejus_required_age', 'degree', 'component_id', 'projection_degree', 'projection_weighted_degree', 'projection_harmonic_centrality', 'projection_betweenness_centrality', 'leiden_1_5_res_community', 'leiden_0_5_res_community']
Edge attributes: ['weight']


In [17]:
g.write_graphml(
    "../hardrive_mount/projected_graph_with_names_and_communities_filtered_10k_games_and_100k_edges.graphml")

## Load examples of each detected community

In [3]:
print("Loading games (nodes) from database with balanced community sampling...")

# First, get the top 10 communities by size
community_sizes_query = """
SELECT 
  leiden_0_5_res_community,
  COUNT(*) as community_size
FROM games
GROUP BY leiden_0_5_res_community
ORDER BY community_size DESC
LIMIT 10
"""

community_sizes_df = pd.read_sql(community_sizes_query, engine)
top_communities = community_sizes_df['leiden_0_5_res_community'].tolist()

print(f"Top 10 communities by size:")
print(community_sizes_df)

# Load 1000 games from each of the top 10 communities
games_query = """
WITH ranked_games AS (
  SELECT 
    game_id,
    name,
    steam_appid,
    price_currency,
    price_final,
    price_initial,
    categories_ids,
    genre_ids,
    release_date,
    rating_dejus_rating,
    rating_dejus_required_age,
    degree,
    component_id,
    projection_degree,
    projection_weighted_degree,
    projection_harmonic_centrality,
    projection_betweenness_centrality,
    leiden_1_5_res_community,
    leiden_0_5_res_community,
    ROW_NUMBER() OVER (PARTITION BY leiden_0_5_res_community ORDER BY projection_weighted_degree DESC) as rn
  FROM games
  WHERE leiden_0_5_res_community = ANY(%s)
)
SELECT 
  game_id,
  name,
  steam_appid,
  price_currency,
  price_final,
  price_initial,
  categories_ids,
  genre_ids,
  release_date,
  rating_dejus_rating,
  rating_dejus_required_age,
  degree,
  component_id,
  projection_degree,
  projection_weighted_degree,
  projection_harmonic_centrality,
  projection_betweenness_centrality,
  leiden_1_5_res_community,
  leiden_0_5_res_community
FROM ranked_games
WHERE rn <= 1000
ORDER BY leiden_0_5_res_community, projection_weighted_degree DESC
"""

games_df = pd.read_sql(games_query, engine, params=(top_communities,))

print(f"\n✓ Loaded {len(games_df):,} games (nodes)")
print(f"  Memory usage: {games_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nGames per community:")
print(games_df.groupby('leiden_0_5_res_community').size())
print("\nSample of games data:")
print(games_df.head())
print("\nGames data info:")
print(games_df.info())

Loading games (nodes) from database with balanced community sampling...
Top 10 communities by size:
   leiden_0_5_res_community  community_size
0                         0            9862
1                         1            1052
2                         2             742
3                         3             620
4                         4             585
5                         5             551
6                         6             379
7                         7             363
8                         8             348
9                         9             317

✓ Loaded 5,905 games (nodes)
  Memory usage: 3.71 MB

Games per community:
leiden_0_5_res_community
0    1000
1    1000
2     742
3     620
4     585
5     551
6     379
7     363
8     348
9     317
dtype: int64

Sample of games data:
   game_id                    name  steam_appid price_currency  price_final  \
0   292030  Ведьмак 3: Дикая Охота       292030            KZT    1199900.0   
1  1086940         Ba

In [4]:
print("Loading game projection edges from database for the sampled games...")

# Get the list of game_ids from the loaded games
game_ids = games_df['game_id'].tolist()

# Load edges where both games are in our sampled set
edges_query = """
SELECT 
  game_id_1,
  game_id_2,
  weight
FROM game_projection_edges
WHERE game_id_1 = ANY(%s)
  AND game_id_2 = ANY(%s)
ORDER BY weight DESC
LIMIT 10000
"""

edges_df = pd.read_sql(edges_query, engine, params=(game_ids, game_ids))

print(f"✓ Loaded {len(edges_df):,} edges")
print(
    f"  Memory usage: {edges_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nSample of edges data:")
print(edges_df.head())
print("\nEdge weight statistics:")
print(edges_df['weight'].describe())

Loading game projection edges from database for the sampled games...
✓ Loaded 10,000 edges
  Memory usage: 1.12 MB

Sample of edges data:
  game_id_1 game_id_2  weight
0   1245620    374320   71839
1   1091500    292030   64816
2   1086940   1245620   64010
3   1086940    292030   60402
4   1086940   1091500   57664

Edge weight statistics:
count    10000.000000
mean      3723.048000
std       3430.025161
min       1737.000000
25%       2078.000000
50%       2686.500000
75%       4007.000000
max      71839.000000
Name: weight, dtype: float64


In [5]:
print("Building igraph from database data...")

# Create a mapping from game_id to vertex index
game_id_to_idx = {game_id: idx for idx,
                  game_id in enumerate(games_df['game_id'])}

# Create list of edges using vertex indices
edge_list = []
edge_weights = []

for _, row in edges_df.iterrows():
    game1_idx = game_id_to_idx.get(row['game_id_1'])
    game2_idx = game_id_to_idx.get(row['game_id_2'])

    if game1_idx is not None and game2_idx is not None:
        edge_list.append((game1_idx, game2_idx))
        edge_weights.append(row['weight'])

# Create the igraph Graph
g = ig.Graph(n=len(games_df), edges=edge_list, directed=False)

# Add edge weights
g.es['weight'] = edge_weights

# Add vertex attributes from games dataframe
for col in games_df.columns:
    g.vs[col] = games_df[col].tolist()

print(f"\n✓ Graph constructed successfully!")
print(f"  Vertices: {g.vcount():,}")
print(f"  Edges: {g.ecount():,}")
print(f"  Directed: {g.is_directed()}")
print(f"  Weighted: {'weight' in g.es.attributes()}")
print(f"\nVertex attributes: {g.vs.attributes()}")
print(f"Edge attributes: {g.es.attributes()}")

g.write_graphml(
    "../hardrive_mount/projected_graph_with_names_and_communities_filtered_1k_games_per_community.graphml")

Building igraph from database data...

✓ Graph constructed successfully!
  Vertices: 5,905
  Edges: 10,000
  Directed: False
  Weighted: True

Vertex attributes: ['game_id', 'name', 'steam_appid', 'price_currency', 'price_final', 'price_initial', 'categories_ids', 'genre_ids', 'release_date', 'rating_dejus_rating', 'rating_dejus_required_age', 'degree', 'component_id', 'projection_degree', 'projection_weighted_degree', 'projection_harmonic_centrality', 'projection_betweenness_centrality', 'leiden_1_5_res_community', 'leiden_0_5_res_community']
Edge attributes: ['weight']
